# Sprint 4 — Webinar 15 (Teórico) — Student Version

**Tema:** Funnels, A/B testing y simulación de impacto en revenue  
**Herramienta:** SQLite Online  
**Modalidad:** teoría guiada + práctica acompañada

### Propósito de la sesión
Hoy vas a conectar tres niveles de análisis:

1. **Comportamiento del usuario** dentro del funnel.  
2. **Comparación entre variantes** en un A/B test.  
3. **Traducción del cambio en conversión** a una métrica de negocio como revenue.

### Mi objetivo personal para esta sesión
____________________________________________________________  
____________________________________________________________

## Agenda de trabajo

| Bloque | Tiempo estimado | Qué debo lograr |
|---|---:|---|
| 0. Objetivos y setup | 10 min | |
| 1. Recordatorio teórico | 10 min | |
| 2. Exploración inicial | 15 min | |
| 3. Conversión y simulación | 25 min | |
| 4. Impacto en revenue | 15 min | |
| 5. Segmentación | 10 min | |
| 6. Comunicación a stakeholders | 10 min | |
| 7. Cierre | 5 min | |

**Pregunta guía de la sesión:**  
¿Cómo paso de una mejora en una etapa del funnel a una estimación de impacto económico?

## 0. Objetivos de aprendizaje

Completa la última columna al final de la sesión.

| Objetivo | ¿Lo logré? | En mis palabras |
|---|---|---|
| Explicar qué representa cada etapa del funnel | ☐ Sí / ☐ Parcial / ☐ No | |
| Comparar métricas entre variante A y B | ☐ Sí / ☐ Parcial / ☐ No | |
| Calcular tasas de conversión con SQL | ☐ Sí / ☐ Parcial / ☐ No | |
| Simular una mejora en una etapa del funnel | ☐ Sí / ☐ Parcial / ☐ No | |
| Estimar el impacto en compras y revenue | ☐ Sí / ☐ Parcial / ☐ No | |
| Comunicar resultados a stakeholders de forma simple | ☐ Sí / ☐ Parcial / ☐ No | |

## 0.1. Cómo usar este notebook con SQLite Online

Marca lo que ya verificaste antes de avanzar:

- ☐ Abrí SQLite Online
- ☐ Entiendo que primero debo ejecutar el bloque de creación de tablas
- ☐ Luego debo ejecutar los inserts
- ☐ Tengo claro que hoy trabajaré sobre un funnel con variantes A/B
- ☐ Sé que varias consultas las completaré yo

### Registro de errores
Completa esta tabla cuando algo falle.

| Bloque / consulta | Error observado | Posible causa | ¿Cómo lo corregí? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

## 1. Recordatorio teórico — Funnels y A/B testing

### Mis apuntes rápidos
Completa con tus palabras.

- Un **funnel** representa: ______________________________________________  
- Una **etapa del funnel** mide: ________________________________________  
- Un **A/B test** sirve para: ____________________________________________  
- La **variante A** normalmente representa: ______________________________  
- La **variante B** normalmente representa: ______________________________

### Idea clave
No basta con mirar compras finales. También conviene revisar:
- qué pasa en cada etapa,
- dónde se pierde más gente,
- y si la diferencia cambia por segmento.

## 1.3 Del funnel al impacto de negocio

### Completa la lógica
Si una mejora aumenta una tasa intermedia del funnel, entonces podría ocurrir que:

1. aumente el número de __________________________________________  
2. eso genere más ________________________________________________  
3. y finalmente impacte __________________________________________

### Preguntas guía
1. ¿Por qué una mejora en `begin_checkout` no garantiza automáticamente más revenue?  
   ____________________________________________________________

2. ¿Qué supuestos estás haciendo cuando simulas un escenario futuro?  
   ____________________________________________________________

3. ¿Qué otras variables podrían afectar el resultado además de la conversión?  
   ____________________________________________________________

In [ ]:
-- Setup del dataset para la sesión.
-- Ejecuta este bloque primero.

-- ==============================================
-- DDL S4W15 para SQLiteOnline (sin esquemas)
-- Pega y ejecuta este bloque primero.
-- ==============================================

-- 1) Asegura FK activas en SQLite
PRAGMA foreign_keys = ON;

-- 2) Limpieza idempotente (por si vuelves a correr el script)
DROP TABLE IF EXISTS funnel_sessions;
DROP TABLE IF EXISTS users;

-- 3) Tabla de usuarios (segmentos)
CREATE TABLE users (
  user_id   INTEGER PRIMARY KEY,
  segment   TEXT NOT NULL,       -- 'web' o 'mobile'
  country   TEXT NOT NULL        -- por simplicidad, pocos países
);

-- 4) Tabla de sesiones de funnel
-- Cada fila ~ 1 sesión de usuario en el sitio
CREATE TABLE funnel_sessions (
  session_id        INTEGER PRIMARY KEY,
  user_id           INTEGER NOT NULL,
  variant           TEXT NOT NULL,    -- 'A' (control) o 'B' (tratamiento)
  viewed_product    INTEGER NOT NULL, -- 1 si vio ficha de producto
  added_to_cart     INTEGER NOT NULL, -- 1 si agregó al carrito
  begin_checkout    INTEGER NOT NULL, -- 1 si inició checkout
  purchased         INTEGER NOT NULL, -- 1 si completó compra
  revenue           REAL NOT NULL,    -- importe de la compra (0 si no compró)
  session_date      TEXT NOT NULL,    -- ISO-8601: '2025-10-01'
  FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
);

-- Nota de modelado:
-- En un sistema real tendríamos tabla de eventos a nivel click/etapa.
-- Aquí usamos flags 0/1 por etapa para simplificar el análisis en clase.


In [ ]:
-- Carga de usuarios.
-- Ejecuta este bloque después del DDL.

-- ==============================================
-- Seed de usuarios (segmentos)
-- Ejecuta este bloque después del DDL.
-- ==============================================

INSERT INTO users (user_id, segment, country) VALUES
  (1,  'web',    'CO'),
  (2,  'web',    'CO'),
  (3,  'web',    'MX'),
  (4,  'web',    'MX'),
  (5,  'web',    'AR'),
  (6,  'web',    'CO'),
  (7,  'mobile', 'CO'),
  (8,  'mobile', 'CO'),
  (9,  'mobile', 'MX'),
  (10, 'mobile', 'MX'),
  (11, 'mobile', 'AR'),
  (12, 'mobile', 'CO'),
  (13, 'web',    'CO'),
  (14, 'web',    'CO'),
  (15, 'mobile', 'CO'),
  (16, 'mobile', 'CO'),
  (17, 'web',    'MX'),
  (18, 'mobile', 'MX'),
  (19, 'web',    'AR'),
  (20, 'mobile', 'AR');


In [ ]:
-- Carga de sesiones del funnel.
-- Ejecuta este bloque después del bloque de usuarios.

-- ==============================================
-- Seed de sesiones de funnel
-- Supone un mes de observación con A/B test en marcha.
-- ==============================================

INSERT INTO funnel_sessions (
  session_id, user_id, variant,
  viewed_product, added_to_cart, begin_checkout, purchased,
  revenue, session_date
) VALUES
  -- Variante A (control) - segmento web
  ( 1,  1, 'A', 1,1,1,1, 120.0, '2025-10-01'),
  ( 2,  1, 'A', 1,1,1,0,   0.0, '2025-10-05'),
  ( 3,  2, 'A', 1,1,0,0,   0.0, '2025-10-03'),
  ( 4,  2, 'A', 1,0,0,0,   0.0, '2025-10-07'),
  ( 5,  3, 'A', 1,1,1,1,   80.0, '2025-10-02'),
  ( 6,  3, 'A', 1,1,1,0,   0.0, '2025-10-08'),
  ( 7,  4, 'A', 1,1,0,0,   0.0, '2025-10-04'),
  ( 8,  5, 'A', 1,0,0,0,   0.0, '2025-10-06'),
  ( 9,  6, 'A', 1,1,1,1, 150.0, '2025-10-09'),
  (10,  6, 'A', 1,1,1,1, 130.0, '2025-10-12'),

  -- Variante A (control) - segmento mobile
  (11,  7, 'A', 1,1,0,0,   0.0, '2025-10-01'),
  (12,  7, 'A', 1,1,1,1,   90.0, '2025-10-10'),
  (13,  8, 'A', 1,0,0,0,   0.0, '2025-10-03'),
  (14,  9, 'A', 1,1,1,0,   0.0, '2025-10-05'),
  (15, 10, 'A', 1,1,1,1,   60.0, '2025-10-07'),
  (16, 11, 'A', 1,1,0,0,   0.0, '2025-10-09'),

  -- Variante B (tratamiento) - segmento web
  (17,  1, 'B', 1,1,1,1, 140.0, '2025-10-15'),
  (18,  2, 'B', 1,1,1,1, 110.0, '2025-10-16'),
  (19,  3, 'B', 1,1,1,0,   0.0, '2025-10-18'),
  (20,  4, 'B', 1,1,1,1,  95.0, '2025-10-19'),
  (21, 13, 'B', 1,1,1,1, 105.0, '2025-10-20'),
  (22, 14, 'B', 1,1,0,0,   0.0, '2025-10-21'),
  (23, 17, 'B', 1,1,1,0,   0.0, '2025-10-22'),
  (24, 19, 'B', 1,0,0,0,   0.0, '2025-10-23'),

  -- Variante B (tratamiento) - segmento mobile
  (25,  7, 'B', 1,1,1,1,  85.0, '2025-10-15'),
  (26,  8, 'B', 1,1,1,0,   0.0, '2025-10-16'),
  (27,  9, 'B', 1,1,1,1,  70.0, '2025-10-17'),
  (28, 10, 'B', 1,1,1,1,  65.0, '2025-10-18'),
  (29, 11, 'B', 1,1,0,0,   0.0, '2025-10-19'),
  (30, 12, 'B', 1,1,1,1,  95.0, '2025-10-20'),
  (31, 15, 'B', 1,0,0,0,   0.0, '2025-10-21'),
  (32, 16, 'B', 1,1,1,0,   0.0, '2025-10-22'),
  (33, 18, 'B', 1,1,1,1,  75.0, '2025-10-23'),
  (34, 20, 'B', 1,1,1,1,  88.0, '2025-10-24');


## 2. Exploración inicial del dataset

Antes de modelar escenarios, necesito entender qué datos tengo.

### Checklist de exploración
- ☐ Revisé algunas filas completas
- ☐ Confirmé que `variant` solo contiene A y B
- ☐ Verifiqué que las etapas del funnel tengan lógica
- ☐ Identifiqué columnas clave para el análisis

### Recordatorio
Una exploración inicial ayuda a detectar:
- valores raros,
- combinaciones imposibles,
- problemas de calidad,
- o interpretaciones incorrectas del dataset.

### Ejercicio 2.1 — Inspeccionar las primeras filas

### Antes de ejecutar
Anota qué esperas ver en una fila de `funnel_sessions`:

| Columna | ¿Qué representa? | Tipo de valor esperado |
|---|---|---|
| `variant` | | |
| `viewed_product` | | |
| `added_to_cart` | | |
| `begin_checkout` | | |
| `purchased` | | |
| `revenue` | | |
| `session_date` | | |

### Tu tarea
1. Lista las primeras 10 filas de `funnel_sessions`.  
2. Revisa si las combinaciones de flags tienen sentido.  
3. Detecta al menos un patrón que te llame la atención.

### Observaciones
____________________________________________________________  
____________________________________________________________

In [ ]:
-- Ejercicio 2.1
-- Escribe una consulta para inspeccionar las primeras filas.
-- Sugerencia: usa ORDER BY session_date, session_id y LIMIT 10.

SELECT
    *
FROM funnel_sessions
-- completa aquí
;

### Ejercicio 2.2 — Conteo de sesiones por variante y segmento

### Antes de escribir SQL
Responde:

- ¿Por qué es importante saber cuántas observaciones tiene cada grupo?  
  ________________________________________________________

- Si una variante tiene muy pocas sesiones, ¿qué problema podría aparecer?  
  ________________________________________________________

### Tu tarea
Calcula el número de sesiones por combinación de:
- `variant`
- `segment`

### Resultado esperado
| Variante | Segmento | Nº sesiones | ¿Qué concluyo? |
|---|---|---:|---|
| | | | |
| | | | |
| | | | |
| | | | |

In [ ]:
-- Ejercicio 2.2
-- Une funnel_sessions con users y cuenta sesiones por variante y segmento.

SELECT
    -- completa aquí
FROM funnel_sessions fs
JOIN users u USING (user_id)
GROUP BY
    -- completa aquí
ORDER BY
    -- completa aquí
;

## 3. A/B testing en funnels y simulación de mejoras

En esta parte vas a trabajar en dos niveles:

1. **Diagnóstico actual:** medir las conversiones reales por variante.  
2. **Escenario simulado:** suponer una mejora y estimar su efecto.

### Nota metodológica
Simular no significa “adivinar”. Significa:
- partir de datos observados,
- explicitar supuestos,
- y cuantificar un posible impacto.

### Ejercicio 3.1 — Tasas de conversión por variante (Actual)

### Completa primero la lógica del funnel
- `add_to_cart_rate` = ______________________________________  
- `checkout_rate` = _________________________________________  
- `purchase_rate_final` = ___________________________________

### Tu tarea
Para cada variante (`A`, `B`) calcula:

1. `n_view`
2. `n_cart`
3. `n_chk`
4. `n_buy`

Luego calcula las tasas de conversión.

### Registro de resultados
| Variante | n_view | n_cart | n_chk | n_buy | add_to_cart_rate | checkout_rate | purchase_rate_final |
|---|---:|---:|---:|---:|---:|---:|---:|
| A | | | | | | | |
| B | | | | | | | |

### Interpretación
¿En qué etapa parece estar la principal diferencia entre A y B?  
____________________________________________________________

In [ ]:
-- Ejercicio 3.1
-- Calcula los conteos por etapa y luego las tasas por variante.

WITH base AS (
    SELECT
        variant,
        -- completa aquí
    FROM funnel_sessions
    GROUP BY variant
)
SELECT
    variant,
    n_view,
    n_cart,
    n_chk,
    n_buy,
    -- completa las tasas aquí
FROM base
;

### Ejercicio 3.2 — Simular una mejora en una etapa del funnel (Variante B)

### Hipótesis de trabajo
Supón que en la variante **B** mejoras la transición:

`added_to_cart → begin_checkout`

y que esta tasa sube en **+5 puntos porcentuales**.

### Tu tarea
1. Reutiliza los agregados de la variante B.  
2. Calcula `checkout_rate_actual`.  
3. Define `checkout_rate_sim = checkout_rate_actual + 0.05`.  
4. Estima:
   - `n_chk_sim`
   - `n_buy_sim`
   - `delta_buy`

### Antes de escribir SQL
Completa:

- Si mejora `checkout_rate`, entonces deberían aumentar: __________________  
- Para estimar `n_buy_sim`, estoy asumiendo que esta tasa se mantiene constante: __________________

### Registro del escenario
| Variante | checkout_rate_actual | checkout_rate_sim | n_buy actual | n_buy simulado | delta_buy |
|---|---:|---:|---:|---:|---:|
| B | | | | | |

In [ ]:
-- Ejercicio 3.2
-- Trabaja solo con la variante B.
-- Puedes reutilizar un CTE base y luego calcular el escenario simulado.

WITH base AS (
    SELECT
        variant,
        -- completa aquí
    FROM funnel_sessions
    WHERE variant = 'B'
    GROUP BY variant
),
rates AS (
    SELECT
        variant,
        n_view,
        n_cart,
        n_chk,
        n_buy,
        -- completa aquí
    FROM base
)
SELECT
    variant,
    n_view,
    n_cart,
    n_chk,
    n_buy,
    -- completa aquí
FROM rates
;

## 4. Estimando impacto en métricas de negocio

Hasta aquí mediste **compras**.  
Ahora toca traducir el escenario a **revenue**.

### Idea central
Si el ticket promedio se mantiene constante, entonces una variación en compras puede convertirse en una variación estimada de ingresos.

### Fórmulas guía
- `avg_revenue_per_purchase` = __________________________________  
- `revenue_sim` = _______________________________________________  
- `uplift_revenue` = ____________________________________________

### Ejercicio 4.1 — Ticket promedio y revenue actual

### Tu tarea
Para la variante B:

1. Cuenta cuántas sesiones tienen `purchased = 1`.  
2. Suma el `revenue` de esas compras.  
3. Calcula el ticket promedio.

### Registro
| Variante | n_buy | revenue_total | avg_revenue_per_purchase |
|---|---:|---:|---:|
| B | | | |

### Interpretación
¿Qué significa que el ticket promedio permanezca constante en una simulación?  
____________________________________________________________

In [ ]:
-- Ejercicio 4.1
-- Calcula compras, revenue total y ticket promedio para la variante B.

WITH stats_b AS (
    SELECT
        -- completa aquí
    FROM funnel_sessions
    WHERE variant = 'B'
)
SELECT
    -- completa aquí
FROM stats_b
;

### Ejercicio 4.2 — Revenue simulado con el nuevo funnel

### Tu tarea
1. Reutiliza o vuelve a calcular `n_buy_sim`.  
2. Reutiliza el ticket promedio de la variante B.  
3. Calcula:
   - `revenue_actual`
   - `revenue_sim`
   - `uplift_revenue`

### Antes de empezar
¿Qué supuesto hace más simple esta simulación?  
____________________________________________________________

### Registro del resultado
| Variante | n_buy_actual | n_buy_sim | revenue_actual | revenue_sim | uplift_revenue |
|---|---:|---:|---:|---:|---:|
| B | | | | | |

In [ ]:
-- Ejercicio 4.2
-- Une la lógica de conversión simulada con la del ticket promedio.

WITH conv AS (
    SELECT
        'B' AS variant,
        -- completa aquí
    FROM funnel_sessions
    WHERE variant = 'B'
),
rates AS (
    SELECT
        variant,
        n_view,
        n_cart,
        n_chk,
        n_buy,
        -- completa aquí
    FROM conv
),
sim AS (
    SELECT
        variant,
        n_buy,
        -- completa aquí
    FROM rates
),
rev AS (
    SELECT
        'B' AS variant,
        -- completa aquí
    FROM funnel_sessions
    WHERE variant = 'B'
)
SELECT
    -- completa aquí
FROM sim
JOIN rev USING (variant)
;

## 5. Comparando escenarios por segmento

En producto no siempre basta con ver el promedio general.  
Muchas veces el efecto cambia por tipo de usuario.

### Completa esta idea
Analizar por segmento sirve para:
- detectar _____________________________________________________
- priorizar ____________________________________________________
- comunicar ____________________________________________________

### Ejercicio 5.1 — Revenue actual por segmento y variante

### Tu tarea
Para cada combinación `(segment, variant)` calcula:

1. número de sesiones  
2. número de compras  
3. revenue total  
4. ticket promedio

### Registro
| Segmento | Variante | n_sessions | n_buy | revenue_total | avg_revenue_per_purchase | ¿Qué observo? |
|---|---|---:|---:|---:|---:|---|
| | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |

In [ ]:
-- Ejercicio 5.1
-- Une users con funnel_sessions y calcula las métricas por segmento y variante.

SELECT
    -- completa aquí
FROM funnel_sessions fs
JOIN users u USING (user_id)
GROUP BY
    -- completa aquí
ORDER BY
    -- completa aquí
;

### Ejercicio 5.2 — ¿Qué segmento aporta más al uplift simulado?

### Tu tarea
Plantea una estrategia para estimar el uplift por segmento.

Puedes hacerlo:
- primero para un solo segmento (por ejemplo `mobile`),
- y luego extenderlo al resto.

### Plan de trabajo
Completa estos pasos antes de escribir la query:

1. Filtraré la variante: __________________  
2. Filtraré el segmento: __________________  
3. Calcularé primero estas cantidades base: __________________  
4. Luego estimaré: __________________  
5. Finalmente compararé con: __________________

### Conclusión esperada
¿Qué segmento crees, antes de calcularlo, que tendrá mayor uplift y por qué?  
____________________________________________________________

In [ ]:
-- Ejercicio 5.2
-- Usa este espacio para intentar la solución.
-- Puedes empezar con un solo segmento y luego generalizar.

-- Idea de esqueleto:
-- WITH conv AS (
--     SELECT ...
--     FROM funnel_sessions fs
--     JOIN users u USING (user_id)
--     WHERE fs.variant = 'B'
--       AND u.segment = 'mobile'
-- ),
-- rates AS (...),
-- sim AS (...),
-- rev AS (...)
-- SELECT ...
;

## 6. Visualizando funnels para stakeholders (Google Sheets)

Una vez que tienes el análisis, toca **comunicarlo**.

### Ejercicio 6.1 — Tabla de resumen Actual vs Simulado

Construye una tabla manual en Google Sheets con estas columnas:

| Métrica | Actual | Simulado |
|---|---:|---:|
| Compras totales variante B | | |
| Tasa de purchase final B | | |
| Revenue total variante B | | |
| Ticket promedio B | | |

### Checklist
- ☐ La tabla tiene títulos claros
- ☐ Los valores están bien copiados
- ☐ Distingo métricas de volumen, tasa y dinero
- ☐ La tabla se entiende sin necesidad de leer el SQL

### Nota de comunicación
¿Qué dos métricas mostrarías sí o sí a un stakeholder no técnico?  
____________________________________________________________

### Ejercicio 6.2 — Gráfico de columnas Actual vs Simulado

### Tu tarea en Google Sheets
1. Selecciona tu tabla resumen.  
2. Inserta un gráfico de columnas.  
3. Ajusta título, etiquetas y formato.  
4. Escribe una mini conclusión de 3 a 5 líneas.

### Mini borrador de narrativa
Completa esta estructura:

- La simulación asume que ______________________________________  
- Bajo ese supuesto, la variante B pasaría de __________________ a __________________  
- Eso implicaría un uplift estimado de ________________________  
- La principal advertencia metodológica es _____________________

## 7. Cierre y siguientes pasos

### Lo más importante de hoy
Completa con tus palabras:

1. Un funnel sirve para ________________________________________  
2. Un A/B test permite _________________________________________  
3. Una simulación de revenue es útil cuando ____________________  
4. Segmentar resultados ayuda a ________________________________

### Autoevaluación rápida
Marca cómo te sientes al final de la clase:

- ☐ Puedo seguir el análisis si veo una query parecida
- ☐ Puedo explicar el razonamiento aunque todavía me cueste escribir la query
- ☐ Necesito repasar joins, agregaciones y tasas
- ☐ Necesito practicar más la parte de interpretación de negocio

### ¿Necesitas ayuda?

Si te trabas en esta sesión, anota aquí qué necesitas revisar:

- Tema o concepto: _____________________________________________
- Query o bloque que me costó más: _____________________________
- Error más frecuente que tuve hoy: ____________________________
- Duda que quiero llevar a tutoría o Discord: _________________

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Cierre

### Kahoot de repaso / preguntas de salida
Responde con tus palabras:

1. ¿Por qué una mejora en una etapa intermedia del funnel puede o no puede traducirse en más revenue?  
   ________________________________________________________

2. ¿Qué riesgo tiene analizar solo el promedio global y no revisar segmentos?  
   ________________________________________________________

3. ¿Qué supuesto del escenario simulado te parece más débil o más discutible?  
   ________________________________________________________

### Mi principal aprendizaje de hoy
____________________________________________________________  
____________________________________________________________

## Siguientes pasos

- Repite el análisis cambiando el uplift: `+0.03`, `+0.10`, etc.
- Prueba una simulación en otra etapa del funnel.
- Repite la lógica solo para `web` o solo para `mobile`.
- Escribe una recomendación corta para producto con base en tus resultados.

### Tarea personal opcional
La próxima vez quiero llegar sabiendo mejor:  
____________________________________________________________